# Demo: Diffusion-Based Draft Models for Speculative$^2$ Decoding

**CS 5788 Generative Models, Cornell — Cashel Fitzgerald, Tishya Khanna**

This notebook is a thin wrapper around the project's code repository. The actual engine, benchmarking harness, and DFlash integration live under `ssd/` and `bench/` in the repo. Here we only show:

1. The setup needed to reproduce our runs (models, datasets, hardware).
2. The single code change we made to turn standard `dflash_ssd` (V1) into our universal-drafter variant (V2).
3. The exact commands we ran on Modal.
4. The benchmark numbers reported in the written report, with their full metric breakdown.

## 1. Setup

All runs use a custom inference engine (the SSD codebase) extended with a DFlash-backed `dflash_ssd` draft backend. We added a flag `--dflash-universal-drafter` that switches between two regimes.

**Models** (all pulled from Hugging Face):

| Role | Model | Notes |
|---|---|---|
| Verifier (AR target) | `Qwen/Qwen3-8B` | Only does prefill + verify; never emits a draft token. |
| AR drafter | `Qwen/Qwen3-0.6B` | Populates the SSD speculation tree asynchronously. |
| Block-diffusion drafter | `z-lab/Qwen3-8B-DFlash-b16` | Block size 16; conditions on target hidden states. |

**Dataset:** GSM8K (10k samples downloaded via `scripts/get_data_from_hf.py`; first 8 prompts used for the quick subset).

**Hardware:** 2 × NVIDIA H100 80GB on Modal (`gpu='H100:2'`).  Target lives on GPU 0; AR draft and DFlash live on GPU 1 (async S$^2$D pipeline).

**Decoding:** greedy ($T=0$), `k=15` speculative tokens, DFlash block size 16, batch 1, 128 output tokens per prompt.

## 2. What we changed in the engine

The repo already shipped `--draft-backend dflash_ssd` (V1): on a cache **hit** it serves the cached tokens verbatim; on a **miss** it runs DFlash from all-mask. Our project adds the universal-drafter variant (V2): on a hit, *also* run DFlash, but seed the masked block with the cached tokens as a prior.

The relevant code lives in `ssd/engine/draft_backends.py` (`DFlashSeedGenerator.draft_from_token`) and `ssd/engine/draft_runner.py` (`hit_cache_and_respond`). Excerpts below — the **full** implementation is in the repo.

In [ ]:
# ssd/engine/draft_backends.py  —  V2 hook: optional prior_tokens argument
@torch.inference_mode()
def draft_from_token(
    self,
    *,
    recovery_token_id: int,
    start: int,
    lookahead: int,
    seq_id: int | None = None,
    prior_tokens: torch.Tensor | None = None,   # <-- new for V2
):
    """DFlash produces a length-`lookahead` block conditioned on target hidden
    states. If `prior_tokens` is provided (V2 hit path), positions 1..k of the
    masked block are seeded with cached tokens instead of mask tokens."""
    ...
    block_output_ids = torch.full((1, self.block_size), self.mask_token_id, ...)
    block_output_ids[0, 0] = recovery_token_id
    if prior_tokens is not None:
        n = min(prior_tokens.numel(), self.block_size - 1)
        block_output_ids[0, 1:1 + n] = prior_tokens[:n]
    # ... single forward pass through DFlash, return draft tokens + logits ...

In [ ]:
# ssd/engine/draft_runner.py  —  V2 routing inside hit_cache_and_respond
miss_mask = ~cache_hits.to(torch.bool)
hit_mask  = cache_hits.to(torch.bool)

if self.config.dflash_universal_drafter and hit_mask.any():
    # V2: re-run DFlash on hits using the cached continuation as a prior.
    self._dflash_seed_slots(
        slot_mask=hit_mask,
        prior_tokens_per_slot=cached_hit_tokens,
        slot_label='hit',
        ...
    )

if miss_mask.any():
    # V1 + V2: DFlash from all-mask on misses (unchanged).
    self._dflash_seed_slots(
        slot_mask=miss_mask,
        prior_tokens_per_slot=None,
        slot_label='miss',
        ...
    )

We also had to advance the per-sequence DFlash KV cache and target-activation backlog on hits as well as misses, since V2 now consumes activations on both paths (see `ssd/engine/speculator_async.py`).

## 3. How we ran the experiments

Everything went through a single Modal entry point (`modal_run.py`). The local commands were:

```bash
# one-time Modal auth
modal token set --token-id <id> --token-secret <secret> --profile zomma

# V1: dflash_ssd, cached tokens on hits
modal run modal_run.py --num-seqs 8 --output-len 128 --no-universal

# V2: dflash_ssd, DFlash re-drafts on hits with cached prior
modal run modal_run.py --num-seqs 8 --output-len 128
```

Inside the Modal container these translate to:

```bash
python -O bench/bench.py \
    --qwen --size 8 \
    --spec --async --draft-backend dflash_ssd \
    --draft 0.6 --dflash-draft <z-lab/Qwen3-8B-DFlash-b16 snapshot path> \
    --k 15 --gpus 2 --b 1 --temp 0 --dtemp 0 \
    --numseqs 8 --output_len 128
    # add --dflash-universal-drafter to switch from V1 to V2
```

## 4. Results

Below we load the metrics that were emitted by `bench.py` and saved into `results/v1_v2_qwen3_8b_gsm8k.json`. We do not re-run the engine here; these are the actual numbers from our Modal runs.

In [1]:
import json, pathlib
results = json.loads(pathlib.Path('../results/v1_v2_qwen3_8b_gsm8k.json').read_text())
cfg = results['common_config']
print(f"Loaded: {results['experiment']}")
print(f"  Hardware       : {cfg['hardware']}")
print(f"  Prompts x out  : {cfg['num_prompts']} x {cfg['output_len']}")
print(f"  Total tokens   : {cfg['total_tokens']}")

Loaded: SSD + DFlash on Qwen3-8B, GSM8K (greedy, T=0)
  Hardware       : 2x H100 (Modal, gpu='H100:2')
  Prompts x out  : 8 x 128
  Total tokens   : 1024


In [2]:
v1 = results['results']['V1']
v2 = results['results']['V2']
rows = [
    ('Total throughput (tok/s)',          'throughput_tok_per_s_total'),
    ('Decode-only throughput (tok/s)',    'throughput_tok_per_s_decode_only'),
    ('Wall-clock time (s)',               'wall_time_s'),
    ('Avg accepted / step (incl. recovery)', 'avg_tokens_accepted_per_step_incl_recovery'),
    ('Cache hit rate',                    'avg_cache_hit_rate'),
    ('Accepted on cache HIT',             'avg_tokens_accepted_on_hit'),
    ('Accepted on cache MISS',            'avg_tokens_accepted_on_miss'),
    ('Avg draft step time (ms)',          'avg_draft_step_ms'),
]
print(f"  {'':38s}  {'V1 (cached hits)':>16s}  {'V2 (universal drafter)':>22s}")
for label, key in rows:
    print(f"  {label:38s}  {v1[key]:>16.2f}  {v2[key]:>22.2f}")

                                  V1 (cached hits)  V2 (universal drafter)
  Total throughput (tok/s)                   77.77                   31.33
  Decode-only throughput (tok/s)            121.00                   37.00
  Wall-clock time (s)                        13.17                   32.69
  Avg accepted / step (incl. recovery)        6.79                    2.10
  Cache hit rate                              0.83                    0.95
  Accepted on cache HIT                       7.36                    1.97
  Accepted on cache MISS                      4.04                    4.37
  Avg draft step time (ms)                   52.65                   53.43


## 5. Qualitative output (lossless against AR target)

Greedy SD rejection sampling guarantees that draft-accepted tokens come from the target's distribution, so V1 and V2 generations are token-for-token identical to plain AR decoding from Qwen3-8B at $T=0$. The example below is the first GSM8K prompt in our run (V1).

In [3]:
print("PROMPT:")
print("  system")
print("  You are a helpful assistant.")
print("  user")
print("  Natalia sold clips to 48 of her friends in April, and then she sold half")
print("  as many clips in May. How many clips did Natalia sell altogether in April")
print("  and May?")
print("  assistant")
print("")
print("GENERATION (V1, T=0):")
print("  Natalia sold clips to **48 friends in April**.")
print("  In **May**, she sold **half as many** clips as in April.")
print("  So, in May, she sold:  48/2 = 24.")
print("  To find the total number of clips sold in April and May, we add:")
print("  48 + 24 = 72.")
print("  ### Final Answer: **72 clips**")

PROMPT:
  system
  You are a helpful assistant.
  user
  Natalia sold clips to 48 of her friends in April, and then she sold half
  as many clips in May. How many clips did Natalia sell altogether in April
  and May?
  assistant

GENERATION (V1, T=0):
  Natalia sold clips to **48 friends in April**.
  In **May**, she sold **half as many** clips as in April.
  So, in May, she sold:  48/2 = 24.
  To find the total number of clips sold in April and May, we add:
  48 + 24 = 72.
  ### Final Answer: **72 clips**


## 6. Take-away

- **V1 (`dflash_ssd` with cached hits)** reproduces the DFlash paper's acceptance length $(\tau = 6.79$ vs paper $\tau = 6.54$ on Qwen3-8B GSM8K), confirming our DFlash integration is correct.
- **V2 (universal drafter)** collapses hit-path acceptance from 7.36 to 1.97. The reason is that DFlash is trained on blocks of the form $[\text{anchor}, \text{mask}, \text{mask}, \ldots]$, so feeding cached tokens as a non-mask prior is out-of-distribution input. 95% of steps go through the hit path, dragging total throughput down 2.5×.
- **Headline number:** V1 reaches **77.77 tok/s** on Qwen3-8B + 2× H100 with $\tau = 6.79$ accepted tokens / step, lossless against the AR target. Closing the gap to the DFlash paper's 1175 tok/s (B200/SGLang) would require running inside SGLang's production engine on B200 — those gains are orthogonal to our contribution.